# Exploring PET enzymes with a pinned release

This notebook is **not executed** when the documentation is built (the site
makes no live calls); run it yourself with `pip install "plasticdb[pandas,bio]" matplotlib`.
Replace the release with one from `client.releases.list()`.

In [ ]:
import plasticdb

RELEASE = "2026.50"
client = plasticdb.Client(release=RELEASE, cache_dir="~/.cache/plasticdb")
client

## Which release answered?

In [ ]:
client.provenance().to_dict()

## Wild-type PET enzymes

Lists are lazy: `total` is one request, `to_pandas()` fetches every page.

In [ ]:
pet = client.proteins.list(plastic="PET", kind="wild_type")
print(pet.total)
df = pet.to_pandas()
df[["accession", "name", "ec_number", "year", "source_microorganism.name"]].head()

In [ ]:
df.groupby("year").size().plot.bar(title="PET enzymes by year of first report");

In [ ]:
df["source_microorganism.name"].value_counts().head(10).plot.barh();

## Evidence for one enzyme

Interaction, experiment, observation: one row per measured value at the deepest level.

In [ ]:
accession = df["accession"].dropna().iloc[0]
evidence = client.evidence.interactions(accession)
evidence.to_pandas(level="observation").head()

## Sequences

In [ ]:
pet.to_fasta("pet_enzymes.fasta")  # also written to disk
records = pet.to_biopython()  # Bio.SeqRecord list
len(records)

## Record what you used

In [ ]:
print(client.cite())

Search, statistics and the change feed are live only, so they need an unpinned client
(see [Reproducibility](../reproducibility.md#what-cannot-be-pinned)):

```python
live = plasticdb.Client()
live.search("cutinase").first()
```